# 03 · Preparar textos pt/en e entradas de extração

**CPU · Sem download de modelos.** Execute depois da seleção no notebook 02. Este caderno valida a amostra, preserva os textos originais, aplica normalização mínima e prepara arquivos JSONL para uso posterior pelo Qwen3-8B.

Os idiomas do título e do resumo são registrados separadamente. São aceitos português, inglês e pares mistos pt/en. Não há tradução automática antes da extração. Os termos e trechos de suporte devem permanecer no idioma em que aparecem.

O schema de saída organiza memes técnicos, pares de convergência, funções, materiais, processos e efeitos. A futura geração deverá guardar o retorno bruto, validar a estrutura e confirmar as evidências literais. Uma resposta inválida será registrada como falha, nunca como ausência de evidência.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
PASTA_DRIVE = "/content/drive/MyDrive/Replicação LLM EIDT IC"  # @param {type:"string"}
raiz = Path(PASTA_DRIVE).expanduser().resolve(strict=True)
ponto_de_montagem = Path("/content/drive").resolve(strict=True)
if raiz == ponto_de_montagem or not raiz.is_relative_to(ponto_de_montagem) or not raiz.is_dir():
    raise ValueError("Selecione a pasta completa do projeto dentro do Drive montado.")
print("Pasta selecionada:", raiz)

## 1. Conferir configuração, schema, prompt e programa

Os rótulos `cpc_proxy` são alvos externos para a adaptação CPC. Eles ficam fora dos prompts da extração. Treino, validação e teste mantêm os mesmos IDs definidos no notebook 02.

In [ ]:
import hashlib
import json
import re
import subprocess
import sys

def sha256_arquivo(caminho):
    resumo = hashlib.sha256()
    with caminho.open("rb") as fluxo:
        for bloco in iter(lambda: fluxo.read(1024 * 1024), b""):
            resumo.update(bloco)
    return resumo.hexdigest()

def conferir_integridade(nomes):
    indice = raiz / "SHA256SUMS.txt"
    assinaturas = {}
    for linha in indice.read_text(encoding="utf-8-sig").splitlines():
        correspondencia = re.fullmatch(r"([0-9a-f]{64})  (.+)", linha)
        if correspondencia:
            nome = correspondencia.group(2)
            if nome in assinaturas:
                raise ValueError(f"Hash duplicado no índice para {nome}.")
            assinaturas[nome] = correspondencia.group(1)
    for nome in nomes:
        caminho = raiz / nome
        if nome not in assinaturas or not caminho.is_file() or sha256_arquivo(caminho) != assinaturas[nome]:
            raise ValueError(f"Falha de integridade em {nome}. Recopie o arquivo e o índice da mesma entrega.")
        print("Conferido:", nome)

def caminho_no_projeto(relativo):
    relativo = Path(relativo)
    if relativo.is_absolute() or ".." in relativo.parts:
        raise ValueError("O caminho configurado deve ser relativo à pasta do projeto.")
    caminho = (raiz / relativo).resolve()
    if not caminho.is_relative_to(raiz):
        raise ValueError("O caminho configurado saiu da pasta do projeto.")
    return caminho

conferir_integridade(("config/experimento.json", "scripts/preprocessar_patstat_br.py",
                     "schemas/extracao_patstat_br.schema.json", "prompts/extracao_patstat_br.txt"))
arquivo_config = raiz / "config/experimento.json"
config = json.loads(arquivo_config.read_text(encoding="utf-8-sig"))
if config.get("piloto", {}).get("id") != "br_patstat_200_qwen3_8b_v1":
    raise ValueError("Atualize o projeto para a versão do piloto brasileiro.")
piloto = config["piloto"]
entrada = caminho_no_projeto(piloto["amostragem"]["saida"])
arquivo_auditoria_selecao = caminho_no_projeto(piloto["amostragem"]["auditoria"])
if not entrada.is_file() or not arquivo_auditoria_selecao.is_file():
    raise FileNotFoundError("Execute o notebook 02 antes de preparar os textos.")
auditoria_selecao = json.loads(arquivo_auditoria_selecao.read_text(encoding="utf-8"))
if sha256_arquivo(entrada) != auditoria_selecao["sha256_piloto"]:
    raise ValueError("A amostra mudou após a seleção. Confira os arquivos do notebook 02.")
saida = caminho_no_projeto(piloto["preprocessamento"]["saida"])
print("Entrada conferida:", entrada.relative_to(raiz))

## 2. Disponibilizar o validador JSON

Esta etapa instala somente `jsonschema==4.25.1` quando a versão estiver ausente ou diferente. O programa de pré-processamento roda em um processo separado, usando a versão instalada. As bibliotecas de GPU serão tratadas no notebook 01 quando necessárias.

In [ ]:
from importlib import metadata

versao_esperada = "4.25.1"
try:
    versao_atual = metadata.version("jsonschema")
except metadata.PackageNotFoundError:
    versao_atual = None
if versao_atual != versao_esperada:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
         "--progress-bar", "off", f"jsonschema=={versao_esperada}"],
        check=True,
    )
print("Validador JSON:", metadata.version("jsonschema"))

## 3. Preparar os arquivos sem alterar as partições

A saída conserva o título e o resumo originais nos metadados. A normalização Unicode NFC e de quebras de linha é mínima. Na extração futura, cada evidência deverá indicar o campo, o idioma e os índices de início e fim no texto normalizado, mantendo a relação com o original preservado.

Os dados das patentes são delimitados como conteúdo de entrada. Qualquer instrução existente em título ou resumo deve ser tratada como texto da patente. O pré-processamento não realiza inferência nem inventa elementos técnicos.

In [ ]:
subprocess.run(
    [sys.executable, str(raiz / "scripts/preprocessar_patstat_br.py"),
     "--entrada", str(entrada), "--saida", str(saida), "--config", str(arquivo_config)],
    cwd=raiz,
    check=True,
)
print("Pré-processamento concluído em:", saida.relative_to(raiz))

## 4. Conferir as saídas e o relatório

Cada JSONL contém um objeto por linha. As entradas de extração e os metadados têm 200 linhas. Os alvos CPC são separados em 160, 20 e 20. A auditoria documenta assinaturas, idiomas e a normalização aplicada. A reexecução deve produzir os mesmos arquivos.

In [ ]:
arquivos_esperados = {
    "entradas_extracao.jsonl": 200,
    "metadados.jsonl": 200,
    "treino_cpc.jsonl": 160,
    "validacao_cpc.jsonl": 20,
    "teste_cpc.jsonl": 20,
}
for nome, total_esperado in arquivos_esperados.items():
    caminho = saida / nome
    with caminho.open(encoding="utf-8") as arquivo:
        registros = [json.loads(linha) for linha in arquivo if linha.strip()]
    if len(registros) != total_esperado:
        raise RuntimeError(f"Contagem inesperada em {nome}: {len(registros)}.")
    print(nome, "·", len(registros), "registros · SHA-256", sha256_arquivo(caminho))
arquivo_auditoria = saida / "auditoria_preprocessamento.json"
auditoria = json.loads(arquivo_auditoria.read_text(encoding="utf-8"))
print(json.dumps(auditoria, ensure_ascii=False, indent=2))
print("Relatório:", arquivo_auditoria.relative_to(raiz))

## 5. Ponto de controle antes da GPU

Os arquivos desta etapa estão preparados para a implementação e validação do ajuste CPC e da extração. Guarde a amostra, a auditoria, os arquivos normalizados e as revisões da configuração e do prompt.

O treino LoRA BF16 do Qwen3-8B continua dependendo de uma GPU compatível. A sessão anterior, com 14,46 GiB livres e sem BF16 nativo, não atende a esse perfil. Uma configuração QLoRA deverá ser registrada como nova decisão de execução antes de treinar. O alinhamento multilíngue e as métricas também serão implementados e validados nas próximas etapas.

A ausência de identificadores de origem para cada texto, de edição/data da extração PATSTAT, de citações e de histórico anterior a 2010 permanece registrada como limitação. Os arquivos preparados aqui não resolvem essas lacunas.